# Part 4: Data Visualization & Machine Learning
## Theme: Student Performance Analysis & Prediction
**Author:** Gaurav Anand Shukla &nbsp;|&nbsp; **ID:** BITSoM_BA_25111017

---
### Overview
This notebook analyses a student performance dataset, produces meaningful visualizations using **Matplotlib** and **Seaborn**, and builds a **Logistic Regression** classifier to predict whether a student will pass or fail — end to end.

**Dataset:** `students.csv` — 15 students · 5 subjects · attendance % · study hours → pass/fail

> **Before submitting:** Go to **Kernel → Restart & Run All**, save the notebook, then commit. All plot outputs must be visible inline.

---
## Imports

In [ ]:
# Standard data science stack — all must be installed before running
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import accuracy_score

%matplotlib inline
print('All libraries imported successfully.')

---
## Task 1 — Data Exploration with Pandas *(5 marks)*

In [ ]:
# Load the dataset from students.csv
df = pd.read_csv('students.csv')
print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

In [ ]:
# 1. First 5 rows using .head()
print('=== First 5 Rows ===')
df.head()

In [ ]:
# 2. Shape and data type of each column using .dtypes
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nData type of each column:')
print(df.dtypes)

In [ ]:
# 3. Summary statistics for all numeric columns using .describe()
print('=== Summary Statistics (mean, min, max, std) ===')
df.describe()

In [ ]:
# 4. Count of students who passed and who failed
counts = df['passed'].value_counts()
print('=== Pass / Fail Counts ===')
print(f"  Passed : {counts.get(1, 0)}")
print(f"  Failed : {counts.get(0, 0)}")

In [ ]:
# 5. Average score per subject for passing vs failing students
# No groupby needed — filter directly using boolean indexing
subject_cols = ['math', 'science', 'english', 'history', 'pe']

pass_avg = df[df['passed'] == 1][subject_cols].mean().round(2)
fail_avg = df[df['passed'] == 0][subject_cols].mean().round(2)

print('Average score per subject — Passing students:')
print(pass_avg)
print('\nAverage score per subject — Failing students:')
print(fail_avg)

In [ ]:
# 6. Student with the highest overall average across all 5 subjects
# Compute a temporary avg_score column using .mean(axis=1)
subject_cols     = ['math', 'science', 'english', 'history', 'pe']
df['avg_score']  = df[subject_cols].mean(axis=1)

top_idx     = df['avg_score'].idxmax()
top_student = df.loc[top_idx, 'name']
top_avg     = round(df.loc[top_idx, 'avg_score'], 2)

print(f'Student with highest overall average: {top_student}  (avg: {top_avg})')

---
## Task 2 — Data Visualization with Matplotlib *(8 marks)*

Five plots — each has a **descriptive title**, **labelled axes**, a **legend** where applicable,
is **saved as a `.png` file**, and is **displayed inline**.

In [ ]:
# Ensure avg_score column exists before any plots
subject_cols    = ['math', 'science', 'english', 'history', 'pe']
df['avg_score'] = df[subject_cols].mean(axis=1)
print('avg_score column is ready for all Task 2 plots.')

### Plot 1 — Bar Chart: Average score per subject (across all students)

In [ ]:
subject_cols    = ['math', 'science', 'english', 'history', 'pe']
avg_per_subject = df[subject_cols].mean()

plt.figure(figsize=(8, 5))
bars = plt.bar(
    subject_cols, avg_per_subject,
    color=['steelblue', 'darkorange', 'mediumseagreen', 'tomato', 'mediumpurple'],
    edgecolor='black'
)
plt.title('Average Score per Subject (All Students)', fontsize=14, fontweight='bold')
plt.xlabel('Subject', fontsize=12)
plt.ylabel('Average Score', fontsize=12)
plt.ylim(0, 110)

# Annotate each bar with its value
for bar, val in zip(bars, avg_per_subject):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val:.1f}', ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.savefig('plot1_bar.png', dpi=100)
plt.show()
print('Saved: plot1_bar.png')

### Plot 2 — Histogram: Distribution of Math scores (5 bins)

In [ ]:
mean_math = df['math'].mean()

plt.figure(figsize=(7, 5))
plt.hist(df['math'], bins=5, color='coral', edgecolor='black', alpha=0.85)
plt.axvline(
    mean_math, color='navy', linestyle='--', linewidth=2,
    label=f'Mean: {mean_math:.1f}'
)
plt.title('Distribution of Math Scores', fontsize=14, fontweight='bold')
plt.xlabel('Math Score', fontsize=12)
plt.ylabel('Number of Students', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plot2_hist.png', dpi=100)
plt.show()
print('Saved: plot2_hist.png')

### Plot 3 — Scatter Plot: Study hours vs Average score (coloured by Pass/Fail)

In [ ]:
# Plot the two groups separately using two plt.scatter() calls
pass_df = df[df['passed'] == 1]
fail_df = df[df['passed'] == 0]

plt.figure(figsize=(8, 5))
plt.scatter(
    pass_df['study_hours_per_day'], pass_df['avg_score'],
    color='green', s=100, edgecolor='black', label='Pass'
)
plt.scatter(
    fail_df['study_hours_per_day'], fail_df['avg_score'],
    color='red', s=100, edgecolor='black', label='Fail'
)
plt.title('Study Hours per Day vs Average Score', fontsize=14, fontweight='bold')
plt.xlabel('Study Hours per Day', fontsize=12)
plt.ylabel('Average Score', fontsize=12)
plt.legend(title='Result', fontsize=11)
plt.tight_layout()
plt.savefig('plot3_scatter.png', dpi=100)
plt.show()
print('Saved: plot3_scatter.png')

### Plot 4 — Box Plot: Attendance % distribution for Pass vs Fail

In [ ]:
# Separate attendance lists for passing and failing students
pass_attendance = df[df['passed'] == 1]['attendance_pct'].tolist()
fail_attendance = df[df['passed'] == 0]['attendance_pct'].tolist()

plt.figure(figsize=(7, 5))
bp = plt.boxplot(
    [pass_attendance, fail_attendance],
    tick_labels=['Pass', 'Fail'],
    patch_artist=True
)
# Colour the boxes
bp['boxes'][0].set_facecolor('lightgreen')
bp['boxes'][1].set_facecolor('lightcoral')

plt.title('Attendance % Distribution: Pass vs Fail', fontsize=14, fontweight='bold')
plt.xlabel('Result', fontsize=12)
plt.ylabel('Attendance (%)', fontsize=12)
plt.tight_layout()
plt.savefig('plot4_box.png', dpi=100)
plt.show()
print('Saved: plot4_box.png')

### Plot 5 — Line Plot: Math and Science scores for every student

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    df['name'], df['math'],
    marker='o', linestyle='-', color='steelblue', linewidth=2, label='Math'
)
plt.plot(
    df['name'], df['science'],
    marker='s', linestyle='--', color='darkorange', linewidth=2, label='Science'
)
plt.title('Math vs Science Scores per Student', fontsize=14, fontweight='bold')
plt.xlabel('Student Name', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.xticks(rotation=45, ha='right')   # rotate 15 names to avoid overlap
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plot5_line.png', dpi=100)
plt.show()
print('Saved: plot5_line.png')

---
## Task 3 — Data Visualization with Seaborn *(4 marks)*

### Plot 6 — Seaborn Bar Plot: Avg Math & Science score by Pass/Fail

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: average math score split by passed
sns.barplot(data=df, x='passed', y='math', ax=axes[0], hue='passed', palette={1:'green',0:'blue'}, legend=False)
axes[0].set_title('Avg Math Score by Pass/Fail', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Passed  (0 = Fail, 1 = Pass)', fontsize=11)
axes[0].set_ylabel('Avg Math Score', fontsize=11)

# Right: average science score split by passed
sns.barplot(data=df, x='passed', y='science', ax=axes[1], palette='Set1')
axes[1].set_title('Avg Science Score by Pass/Fail', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Passed  (0 = Fail, 1 = Pass)', fontsize=11)
axes[1].set_ylabel('Avg Science Score', fontsize=11)

plt.suptitle('Subject Averages by Pass/Fail — Seaborn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot6_seaborn_bar.png', dpi=100)
plt.show()
print('Saved: plot6_seaborn_bar.png')

### Plot 7 — Seaborn Scatter: Attendance % vs Avg Score with regression lines

In [ ]:
plt.figure(figsize=(8, 6))

# Scatter points coloured by pass/fail
sns.scatterplot(
    data=df, x='attendance_pct', y='avg_score',
    hue='passed', palette={1: 'green', 0: 'red'}, s=110
)

# Regression line for passing students
sns.regplot(
    data=df[df['passed'] == 1], x='attendance_pct', y='avg_score',
    scatter=False, color='green', label='Trend (Pass)'
)

# Regression line for failing students
sns.regplot(
    data=df[df['passed'] == 0], x='attendance_pct', y='avg_score',
    scatter=False, color='red', label='Trend (Fail)'
)

plt.title('Attendance % vs Average Score — Seaborn', fontsize=14, fontweight='bold')
plt.xlabel('Attendance (%)', fontsize=12)
plt.ylabel('Average Score', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plot7_seaborn_scatter.png', dpi=100)
plt.show()
print('Saved: plot7_seaborn_scatter.png')

In [ ]:
# ── Seaborn vs Matplotlib — Comparison (required 2–3 sentence comment) ──────
#
# Seaborn made the grouped bar chart noticeably easier: sns.barplot() automatically
# computed group means and added confidence-interval error bars without any manual
# groupby logic — Matplotlib would have required explicit aggregation and careful
# bar positioning. Similarly, sns.regplot() added a regression trend line in a single
# call, whereas Matplotlib would have needed numpy.polyfit() or scipy for the same
# result. On the other hand, Matplotlib gave finer control for the histogram (exact
# bin count, custom mean-line annotation) and the box plot (per-component colour
# styling) — so the two libraries complement each other well: Seaborn excels at
# statistical summaries while Matplotlib is better for fully custom visuals.
#
print('Seaborn vs Matplotlib comparison written in the comment block above.')

---
## Task 4 — Machine Learning with scikit-learn *(8 marks + 2 bonus)*

### Step 1 — Prepare Data

In [ ]:
# Feature columns (exclude 'name' and 'passed' and 'avg_score' from X)
feature_cols = ['math', 'science', 'english', 'history', 'pe',
                'attendance_pct', 'study_hours_per_day']

X = df[feature_cols]   # keep original df intact
y = df['passed']

# 80/20 train-test split with a fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features — fit ONLY on training data, then transform both sets
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Features used    : {feature_cols}')

### Step 2 — Train a Logistic Regression Model

In [ ]:
# Train Logistic Regression on scaled training data
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

train_acc = model.score(X_train_scaled, y_train)
print(f'Training Accuracy : {train_acc * 100:.2f}%')

### Step 3 — Evaluate the Model

In [ ]:
y_pred   = model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy : {test_acc * 100:.2f}%')
print()

# Per-student predictions on the test set
print('Per-student predictions on the test set:')
print(f"{'Name':<12} {'Actual':<10} {'Predicted':<12} Result")
print('-' * 48)

for i in range(len(X_test)):
    idx       = X_test.index[i]
    name      = df.loc[idx, 'name']
    actual    = 'Pass' if y_test.iloc[i]  == 1 else 'Fail'
    predicted = 'Pass' if y_pred[i]       == 1 else 'Fail'
    correct   = '✅ Correct' if y_test.iloc[i] == y_pred[i] else '❌ Wrong'
    print(f'{name:<12} {actual:<10} {predicted:<12} {correct}')

print()
print('Note: With only 3 test samples (15 × 20% = 3), accuracy is 0/33/66/100% — all valid.')

### Step 4 — Feature Importance

In [ ]:
# Extract model coefficients and pair with feature names
coefficients    = model.coef_[0]
feat_importance = sorted(
    zip(feature_cols, coefficients),
    key=lambda x: abs(x[1]), reverse=True
)

print('Feature Importance (sorted by |coefficient|, largest first):')
print(f"{'Feature':<25} {'Coefficient':>12}  Direction")
print('-' * 52)

for feat, coef in feat_importance:
    direction = '→ Pass' if coef > 0 else '→ Fail'
    print(f'{feat:<25} {coef:>+12.4f}  {direction}')

In [ ]:
# Horizontal bar chart — green = pushes toward Pass, red = pushes toward Fail
features_sorted = [f[0] for f in feat_importance]
coefs_sorted    = [f[1] for f in feat_importance]
colours         = ['green' if c > 0 else 'red' for c in coefs_sorted]

plt.figure(figsize=(9, 5))
plt.barh(features_sorted, coefs_sorted, color=colours, edgecolor='black')
plt.axvline(0, color='black', linewidth=1.0)
plt.title(
    'Feature Importance — Logistic Regression Coefficients',
    fontsize=13, fontweight='bold'
)
plt.xlabel('Coefficient Value  (green → Pass  |  red → Fail)', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.savefig('plot8_feature_importance.png', dpi=100)
plt.show()
print('Saved: plot8_feature_importance.png')

### Step 5 — Predict for a New Student *(Bonus — 2 marks)*

In [ ]:
# Feature order must match: math, science, english, history, pe,
#                           attendance_pct, study_hours_per_day
new_student = [[75, 70, 68, 65, 80, 82, 3.2]]

# Scale using the SAME scaler fitted on training data
new_student_scaled = scaler.transform(new_student)

prediction  = model.predict(new_student_scaled)[0]
probability = model.predict_proba(new_student_scaled)[0]

result = 'Pass ✅' if prediction == 1 else 'Fail ❌'

print('New Student Input Features:')
print('  math=75, science=70, english=68, history=65,')
print('  pe=80, attendance=82%, study_hours=3.2/day')
print()
print(f'Predicted Result       : {result}')
print(f'Probability of Passing : {probability[1]:.2%}')
print(f'Probability of Failing : {probability[0]:.2%}')